#### **FastF1 References**

Installation: [https://docs.fastf1.dev/getting_started/installation.html](https://docs.fastf1.dev/getting_started/installation.html)</br>
API Documentation: [https://docs.fastf1.dev/api_reference/index.html](https://docs.fastf1.dev/api_reference/index.html)

In [ ]:
# Uncomment and run to install
# %pip install fastf1

In [ ]:
# Imports
import os

import fastf1 as ff1
import pandas as pd

#### **Event Data (FastF1)**

- Retrieve location data and start times for events from the years 2018-2025
- Only 'Race' data selected, exlcudes Qualifiers


In [ ]:
event_times = []

# Get race starting times and locations for each year
for year in range(2018, 2026):
    event_schedule = ff1.get_event_schedule(year, include_testing=False)
    
    # Loop through season schedule, storing race info
    for _, event in event_schedule.iterrows():
        for i in range(1, 6):
            if event[f'Session{i}'] == 'Race':
                race_start = event[f'Session{i}DateUtc']
                break
        
        event_times.append({
            'EventName': event['EventName'],
            'Country': event['Country'],
            'Location': event['Location'],
            'StartTime': race_start,
            'Year': year
        })
        
event_data = pd.DataFrame(event_times)
    

,EventName,Country,Location,StartTime,Year
0,Australian Grand Prix,Australia,Melbourne,2018-03-25 05:10:00,2018
1,Bahrain Grand Prix,Bahrain,Sakhir,2018-04-08 15:10:00,2018
2,Chinese Grand Prix,China,Shanghai,2018-04-15 06:10:00,2018
3,Azerbaijan Grand Prix,Azerbaijan,Baku,2018-04-29 12:10:00,2018
4,Spanish Grand Prix,Spain,Barcelona,2018-05-13 13:10:00,2018
...,...,...,...,...,...
168,Mexico City Grand Prix,Mexico,Mexico City,2025-10-26 20:00:00,2025
169,São Paulo Grand Prix,Brazil,São Paulo,2025-11-09 17:00:00,2025
170,Las Vegas Grand Prix,United States,Las Vegas,2025-11-23 04:00:00,2025
171,Qatar Grand Prix,Qatar,Lusail,2025-11-30 16:00:00,2025


#### **Location Data**

- Data downloaded from a GitHub repository `f1-curcuits` created by bacinger
- Includes Longitude and Latitude of tracks from 1950 to present
- Location data will be used to aid in weather data collection for each track

Source: [https://github.com/bacinger/f1-circuits](https://github.com/bacinger/f1-circuits)<br>
Locations JSON: [https://github.com/bacinger/f1-circuits/blob/master/f1-locations.json](https://github.com/bacinger/f1-circuits/blob/master/f1-locations.json)

In [46]:
# Import location data for f1 races
url = "https://raw.githubusercontent.com/bacinger/f1-circuits/master/f1-locations.json"

location_data = pd.read_json(url)[['location', 'lon', 'lat']]

# location_data.head()

In [47]:
# Normalize location names across both data sources
location_data['location'] = location_data['location'].replace({
    'Montreal': 'Montréal',
    'Spa Francorchamps': 'Spa-Francorchamps',
    'Sao Paulo': 'São Paulo',
    'Nürburg': 'Nürburgring',
    'Scarperia e San Piero': 'Mugello'
})

event_data['Location'] = event_data['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

# Attach long, lat to each f1 race location
event_location_df = event_data.merge(location_data, left_on='Location', right_on='location', how='left')
event_location_df = event_location_df.drop(columns=['location'])

# Store location data
event_location_df.to_csv('../data/event_location_data.csv', index=False)

#### **Race Data (FastF1)**

Missing Data:
- The first two races of 2018 do not have telemetry data for drivers
- No racing data for Italy Grand Prix 2018

**Note:** May need to rerun multiple times to pull all data (keeps its spot by checking for existing files)<br>
**FastF Rate Limit:** `500 requests per hour`

In [ ]:
# Fetch race data from 2018 to 2025 (Wait and rerun if rate limit exceeded)
for year in range(2018, 2026):
    # Create directories for race data
    for folder in ['laps', 'weather']:
        os.makedirs(f'../data/{year}/{folder}', exist_ok=True)
    
    event_schedule = ff1.get_event_schedule(year, include_testing=False)
    
    # Retrieve race data per event
    for race_number, name in enumerate(event_schedule['EventName'], start=1):
        event = name.lower().replace(' ', '_')

        laps_path    = f'../data/{year}/laps/laps_{event}_{year}.csv'
        weather_path = f'../data/{year}/weather/weather_{event}_{year}.csv'
        
        # Skip if already saved
        if os.path.exists(laps_path) and os.path.exists(weather_path):
            print(f"Skipping {event} {year} — already saved")
            continue
        
        # Write each race to seperate CSV
        try:
            session = ff1.get_session(year, race_number, 'Race')
            session.load(laps=True, weather=True, telemetry=False, messages=False)
            session.laps.to_csv(laps_path, index=False) # lap data
            session.weather_data.to_csv(weather_path, index=False) # weather data
        except: 
            print(f"Failed to find race data for {event} {year}")
        
        # Fetch telemetry data for available drivers
        # for driver in session.drivers:
        #     try:
        #         tel = session.laps.pick_drivers(driver).telemetry
        #         tel.to_csv(f'../data/{year}/telemetry/telemetry_{driver}_{event}_{year}.csv', index=False) # telemetry data
        #     except Exception as e:
        #         print(f"Failed to find driver telemetry for {driver} {event} {year}")